In [1]:
!pip install -q langchain langchain-core langchain-ollama

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

In [5]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 28s (22.8 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [6]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
!ollama --version

In [30]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started successfully")

Ollama server started successfully


In [21]:
!ollama pull llama3.2:3b

In [22]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:3b    a80c4f17acd5    2.0 GB    2 seconds ago    


In [23]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for an engineering student."
)

print(prompt)

input_variables=['topic'] input_types={} partial_variables={} template='Explain {topic} in simple terms for an engineering student.'


In [24]:
llm = OllamaLLM(
    model="llama3.2:3b"
)

print("Ollama LLM created successfully")

Ollama LLM created successfully


In [25]:
parser = StrOutputParser()

chain = prompt | llm | parser

print("LangChain chain created successfully")

LangChain chain created successfully


In [31]:
topics = [
    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Large Language Models",
    "Generative AI"
]

for topic in topics:
    result = chain.invoke({"topic": topic})
    print(f"\nTopic: {topic}")
    print(result)
    print("-" * 60)


Topic: Artificial Intelligence
As an engineering student, you're likely familiar with the basics of programming, algorithms, and data structures. Artificial Intelligence (AI) builds upon these concepts, but with a twist.

**What is Artificial Intelligence?**

Artificial Intelligence refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. Learning from data
2. Reasoning and problem-solving
3. Understanding natural language
4. Making decisions

**Key Components of AI:**

1. **Machine Learning (ML):** A subset of AI that enables computers to learn from data without being explicitly programmed. ML algorithms can improve their performance over time, much like humans do.
2. **Natural Language Processing (NLP):** A subset of AI that deals with processing, understanding, and generating human language.
3. **Computer Vision:** A subset of AI that enables computers to interpret and understand visual data from images and videos.

In [33]:
from langchain_core.messages import HumanMessage, AIMessage

In [34]:
conversation_history = []

def chat_with_memory(user_input):
    history_text = "\n".join(
        [
            f"User: {msg.content}" if isinstance(msg, HumanMessage)
            else f"AI: {msg.content}"
            for msg in conversation_history
        ]
    )

    memory_prompt = PromptTemplate(
        input_variables=["history", "input"],
        template="""
You are a helpful AI assistant.

Conversation history:
{history}

User's new message:
{input}

Answer the user while considering the conversation history.
"""
    )

    memory_chain = memory_prompt | llm | parser

    response = memory_chain.invoke({
        "history": history_text,
        "input": user_input
    })

    conversation_history.append(HumanMessage(content=user_input))
    conversation_history.append(AIMessage(content=response))

    return response

In [35]:
turns = [
    "My name is Bhavana.",
    "I am studying Information Science Engineering.",
    "What branch am I studying?",
    "What is my name?",
    "Suggest an AI project topic suitable for my branch."
]

for i, message in enumerate(turns, 1):

    response = chat_with_memory(message)

    print(f"\nTurn {i}")
    print("User:", message)
    print("AI:", response)
    print("-" * 60)


Turn 1
User: My name is Bhavana.
AI: Hello Bhavana! It's nice to meet you. Since this is the beginning of our conversation, I don't have any information to draw from yet. How can I assist you today? Would you like to ask me a question, share something, or just chat?
------------------------------------------------------------

Turn 2
User: I am studying Information Science Engineering.
AI: Hello Bhavana! It's great to continue our conversation. Since you're studying Information Science Engineering, I'm sure you're diving into the world of technology, data, and innovation.

Considering our conversation history, I'd like to ask: What aspects of Information Science Engineering are you finding most interesting or challenging? Are you focusing on a specific area, such as data science, artificial intelligence, or cybersecurity? Or do you have a particular project or topic that you're working on?

Feel free to share as much or as little as you'd like, and I'll do my best to provide guidance,

In [36]:
print("===== CONVERSATION HISTORY =====\n")

for message in conversation_history:

    if isinstance(message, HumanMessage):
        print("User:", message.content)

    else:
        print("AI:", message.content)

    print()

===== CONVERSATION HISTORY =====

User: My name is Bhavana.

AI: Hello Bhavana! It's nice to meet you. Since this is the beginning of our conversation, I don't have any information to draw from yet. How can I assist you today? Would you like to ask me a question, share something, or just chat?

User: I am studying Information Science Engineering.

AI: Hello Bhavana! It's great to continue our conversation. Since you're studying Information Science Engineering, I'm sure you're diving into the world of technology, data, and innovation.

Considering our conversation history, I'd like to ask: What aspects of Information Science Engineering are you finding most interesting or challenging? Are you focusing on a specific area, such as data science, artificial intelligence, or cybersecurity? Or do you have a particular project or topic that you're working on?

Feel free to share as much or as little as you'd like, and I'll do my best to provide guidance, insights, or just a listening ear.

Use

In [37]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Simulated web search tool."""
    return f"Simulated web search result for: {query}"


@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"


tools = [web_search, calculator]

print("Tools created successfully")
print("1.", web_search.name)
print("2.", calculator.name)

Tools created successfully
1. web_search
2. calculator


In [38]:
print("Web Search Tool:")
print(web_search.invoke("Latest developments in Generative AI"))

print("\nCalculator Tool:")
print(calculator.invoke("125 * 8"))

Web Search Tool:
Simulated web search result for: Latest developments in Generative AI

Calculator Tool:
1000


In [39]:
def simple_agent(task):

    task_lower = task.lower()

    calculation_words = [
        "calculate",
        "multiply",
        "divide",
        "plus",
        "minus",
        "*",
        "/",
        "+"
    ]

    if any(word in task_lower for word in calculation_words):

        expression = task_lower.replace("calculate", "").strip()

        result = calculator.invoke(expression)

        return f"Calculator result: {result}"

    else:

        result = web_search.invoke(task)

        return f"Search result: {result}"

In [40]:
tasks = [
    "Calculate 125 * 8",
    "Calculate 500 / 25",
    "Search for recent developments in Generative AI"
]

for i, task in enumerate(tasks, 1):

    print(f"\nAgent Task {i}")
    print("Input:", task)
    print("Output:", simple_agent(task))
    print("-" * 60)


Agent Task 1
Input: Calculate 125 * 8
Output: Calculator result: 1000
------------------------------------------------------------

Agent Task 2
Input: Calculate 500 / 25
Output: Calculator result: 20.0
------------------------------------------------------------

Agent Task 3
Input: Search for recent developments in Generative AI
Output: Search result: Simulated web search result for: Search for recent developments in Generative AI
------------------------------------------------------------
